# 25c — Report Revision Pre-Adam v3 (Fixed SDP Source Selection)

This notebook fixes the issue found in 25b where the pre-Adam SDP validation outputs could accidentally use an older/broken SDP profile. It explicitly prefers:

`data/processed/sdp_campaign_validation_v2/sdp_campaign_wards_profile_v2.csv`

and only falls back to older files if the v2 profile is unavailable.

It regenerates the pre-Adam pack with corrected SDP metrics, while retaining the caveat recode, party-transition diagnostics and Yorkshire case-study evidence.

## 25c.1 Setup

In [2]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

CAVEAT_DIR = PROCESSED_DIR / "caveat_resolution_v2"
PARTY_DIR = PROCESSED_DIR / "party_transition_diagnostics_v1"
SDP_DIR = PROCESSED_DIR / "sdp_campaign_validation_v2"
YORKS_DIR = PROCESSED_DIR / "yorkshire_sdp_case_study_v1"
OUTPUT_DIR = PROCESSED_DIR / "pre_adam_report_revision_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3


## 25c.2 Helper functions

In [3]:
def find_file(filename, preferred_dirs=None, required=True):
    if preferred_dirs is None:
        preferred_dirs = []
    search_dirs = preferred_dirs + [CAVEAT_DIR, PARTY_DIR, SDP_DIR, YORKS_DIR, PROCESSED_DIR, PROJECT_DIR, Path.cwd()]
    for d in search_dirs:
        p = d / filename
        if p.exists():
            return p
    for d in search_dirs:
        if d.exists():
            matches = list(d.rglob(filename))
            if matches:
                # prefer newest if duplicates exist
                return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(f"Could not find {filename}")
    return None


def read_csv(filename, preferred_dirs=None, required=True):
    p = find_file(filename, preferred_dirs=preferred_dirs, required=required)
    if p is None:
        print("Optional missing:", filename)
        return None, None
    df = pd.read_csv(p, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {p}")
    return df, p


def save(df, filename):
    p = OUTPUT_DIR / filename
    df.to_csv(p, index=False)
    print("Saved:", p, df.shape)
    return p


def num(s):
    return pd.to_numeric(s, errors="coerce")


def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

## 25c.3 Load corrected inputs

In [4]:
# Caveat outputs from 22b
high_conf, _ = read_csv("north_west_high_confidence_review_v2.csv", preferred_dirs=[CAVEAT_DIR])
medium_conf, _ = read_csv("north_west_medium_confidence_review_v2.csv", preferred_dirs=[CAVEAT_DIR])
serious_cav, _ = read_csv("north_west_serious_caveat_manual_review_v2.csv", preferred_dirs=[CAVEAT_DIR])
main_review, _ = read_csv("north_west_reportable_main_review_v2.csv", preferred_dirs=[CAVEAT_DIR])
caveat_summary, _ = read_csv("north_west_caveat_recode_summary_v2.csv", preferred_dirs=[CAVEAT_DIR])

# Party process/transition outputs from 23/25b where available
party_all, _ = read_csv("north_west_party_transition_diagnostics_all_v1.csv", preferred_dirs=[PARTY_DIR], required=False)
party_summary, _ = read_csv("party_transition_summary_by_council_v1.csv", preferred_dirs=[PARTY_DIR], required=False)
process_labels, _ = read_csv("pre_adam_party_process_label_notes_v2.csv", required=False)

# CRITICAL: corrected SDP v2 profile. Prefer v2 validation folder.
sdp, sdp_path = read_csv("sdp_campaign_wards_profile_v2.csv", preferred_dirs=[SDP_DIR])
sdp_counts, _ = read_csv("sdp_candidate_count_check_v2.csv", preferred_dirs=[SDP_DIR], required=False)
sdp_unmatched, _ = read_csv("sdp_unmatched_results_review_v2.csv", preferred_dirs=[SDP_DIR], required=False)

# Yorkshire case study from 24c2
yorks_wards, _ = read_csv("yorkshire_sdp_case_study_wards_v1.csv", preferred_dirs=[YORKS_DIR], required=False)
yorks_tribe, _ = read_csv("yorkshire_sdp_performance_by_tribe_v1.csv", preferred_dirs=[YORKS_DIR], required=False)
yorks_summary, _ = read_csv("yorkshire_case_study_summary_v1.csv", preferred_dirs=[YORKS_DIR], required=False)
middleton, _ = read_csv("middleton_park_case_study_profile_v1.csv", preferred_dirs=[YORKS_DIR], required=False)

print("Using SDP source:", sdp_path)

Loaded north_west_high_confidence_review_v2.csv: (671, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_high_confidence_review_v2.csv
Loaded north_west_medium_confidence_review_v2.csv: (153, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_medium_confidence_review_v2.csv
Loaded north_west_serious_caveat_manual_review_v2.csv: (1, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_serious_caveat_manual_review_v2.csv
Loaded north_west_reportable_main_review_v2.csv: (824, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_reportable_main_review_v2.csv
Loaded north_west_caveat_recode_summary_v2.csv: (4, 6) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_caveat_recode_summary_v2.csv
Loaded north_west_party_transition_diagnostics_all_v1.csv: (825, 22) from c:

## 25c.4 Recompute SDP validation tables from correct v2 source

In [5]:
# Ensure effective vote share exists.
if "sdp_vote_share_effective" not in sdp.columns:
    if "sdp_votes" in sdp.columns and "valid_votes" in sdp.columns:
        sdp["sdp_vote_share_effective"] = num(sdp["sdp_votes"]) / num(sdp["valid_votes"])
    elif "vote_share" in sdp.columns:
        sdp["sdp_vote_share_effective"] = num(sdp["vote_share"])
    else:
        sdp["sdp_vote_share_effective"] = np.nan

if "matched_to_model_v2" not in sdp.columns:
    sdp["matched_to_model_v2"] = sdp.get("WD25CD_model", pd.Series(index=sdp.index)).notna()

# Candidate count check: use existing if loaded, otherwise compute observed counts.
expected_counts = pd.DataFrame({
    "election_year": [2021, 2022, 2023, 2024, 2025, 2026],
    "expected_user_provided": [68, 30, 36, 28, 11, 48],
})
observed = (
    sdp.groupby("election_year", dropna=False)
    .size()
    .reset_index(name="observed_sdp_candidate_rows")
)
candidate_count_check = expected_counts.merge(observed, on="election_year", how="left")
candidate_count_check["observed_sdp_candidate_rows"] = candidate_count_check["observed_sdp_candidate_rows"].fillna(0).astype(int)
candidate_count_check["difference_observed_minus_expected"] = candidate_count_check["observed_sdp_candidate_rows"] - candidate_count_check["expected_user_provided"]
save(candidate_count_check, "pre_adam_sdp_candidate_count_check_v3.csv")

# Performance by year.
by_year = (
    sdp.groupby("election_year", dropna=False)
    .agg(
        sdp_candidate_rows=("result_id", "count") if "result_id" in sdp.columns else ("election_year", "size"),
        matched_rows=("matched_to_model_v2", "sum"),
        total_sdp_votes=("sdp_votes", "sum"),
        mean_sdp_vote_share=("sdp_vote_share_effective", "mean"),
        median_sdp_vote_share=("sdp_vote_share_effective", "median"),
        max_sdp_vote_share=("sdp_vote_share_effective", "max"),
    )
    .reset_index()
)
save(by_year, "pre_adam_sdp_performance_by_year_v3.csv")

# Performance by tribe.
tribe_col = "dominant_cluster_name"
if tribe_col in sdp.columns:
    by_tribe = (
        sdp.groupby(tribe_col, dropna=False)
        .agg(
            sdp_candidate_rows=("election_year", "size"),
            matched_rows=("matched_to_model_v2", "sum"),
            total_sdp_votes=("sdp_votes", "sum"),
            mean_sdp_vote_share=("sdp_vote_share_effective", "mean"),
            median_sdp_vote_share=("sdp_vote_share_effective", "median"),
            max_sdp_vote_share=("sdp_vote_share_effective", "max"),
            mean_model_score=("initial_watchlist_score", "mean") if "initial_watchlist_score" in sdp.columns else ("election_year", "size"),
        )
        .reset_index()
        .sort_values("total_sdp_votes", ascending=False)
    )
else:
    by_tribe = pd.DataFrame()
save(by_tribe, "pre_adam_sdp_performance_by_tribe_v3.csv")

# Performance by latest top party.
party_col = "latest_election_top_party_bucket"
if party_col in sdp.columns:
    by_party = (
        sdp.groupby(party_col, dropna=False)
        .agg(
            sdp_candidate_rows=("election_year", "size"),
            matched_rows=("matched_to_model_v2", "sum"),
            total_sdp_votes=("sdp_votes", "sum"),
            mean_sdp_vote_share=("sdp_vote_share_effective", "mean"),
            median_sdp_vote_share=("sdp_vote_share_effective", "median"),
            max_sdp_vote_share=("sdp_vote_share_effective", "max"),
            mean_model_score=("initial_watchlist_score", "mean") if "initial_watchlist_score" in sdp.columns else ("election_year", "size"),
        )
        .reset_index()
        .sort_values("total_sdp_votes", ascending=False)
    )
else:
    by_party = pd.DataFrame()
save(by_party, "pre_adam_sdp_performance_by_latest_party_v3.csv")

# Highest vote share cases.
case_cols = existing_cols(sdp, [
    "election_year", "council_name", "ward_name", "candidate_name", "sdp_votes", "valid_votes", "sdp_vote_share_effective",
    "WD25NM_model", "LAD25NM_model", "analysis_region", "dominant_cluster_name", "second_cluster_name", "latest_election_top_party_bucket", "initial_watchlist_score", "matched_to_model_v2"
])
highest = sdp.sort_values("sdp_vote_share_effective", ascending=False, na_position="last")[case_cols].head(100)
save(highest, "pre_adam_sdp_highest_vote_share_cases_v3.csv")

# Unmatched rows.
if "matched_to_model_v2" in sdp.columns:
    unmatched = sdp[~sdp["matched_to_model_v2"].astype(bool)].copy()
else:
    unmatched = pd.DataFrame()
save(unmatched, "pre_adam_sdp_unmatched_rows_for_review_v3.csv")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_sdp_candidate_count_check_v3.csv (6, 4)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_sdp_performance_by_year_v3.csv (5, 7)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_sdp_performance_by_tribe_v3.csv (8, 8)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_sdp_performance_by_latest_party_v3.csv (9, 8)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_sdp_highest_vote_share_cases_v3.csv (100, 15)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_sdp_unmatched_rows_for_review_v3.csv (18, 64)


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/pre_adam_report_revision_v3/pre_adam_sdp_unmatched_rows_for_review_v3.csv')

## 25c.5 Refresh headline metrics

In [6]:
headline = {
    "run_date": datetime.now().isoformat(timespec="seconds"),
    "high_confidence_rows": len(high_conf),
    "medium_confidence_rows": len(medium_conf),
    "serious_caveat_rows": len(serious_cav),
    "main_report_rows": len(main_review),
    "sdp_rows_total": len(sdp),
    "sdp_rows_matched_to_model": int(sdp["matched_to_model_v2"].sum()) if "matched_to_model_v2" in sdp.columns else np.nan,
    "sdp_unmatched_rows": len(unmatched),
    "sdp_max_vote_share": float(num(sdp["sdp_vote_share_effective"]).max()),
}

if yorks_wards is not None and len(yorks_wards):
    headline.update({
        "yorkshire_case_study_rows": len(yorks_wards),
        "yorkshire_max_sdp_vote_share": float(num(yorks_wards["max_sdp_vote_share"]).max()) if "max_sdp_vote_share" in yorks_wards.columns else np.nan,
        "yorkshire_mean_sdp_vote_share": float(num(yorks_wards["mean_sdp_vote_share"]).mean()) if "mean_sdp_vote_share" in yorks_wards.columns else np.nan,
    })
if yorks_tribe is not None and len(yorks_tribe):
    headline["yorkshire_top_dominant_tribe"] = yorks_tribe.sort_values("total_sdp_votes", ascending=False).iloc[0]["dominant_cluster_name"]

headline_df = pd.DataFrame([headline])
save(headline_df, "pre_adam_headline_metrics_v3.csv")
headline_df

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_headline_metrics_v3.csv (1, 13)


,run_date,high_confidence_rows,medium_confidence_rows,serious_caveat_rows,main_report_rows,sdp_rows_total,sdp_rows_matched_to_model,sdp_unmatched_rows,sdp_max_vote_share,yorkshire_case_study_rows,yorkshire_max_sdp_vote_share,yorkshire_mean_sdp_vote_share,yorkshire_top_dominant_tribe
0,2026-05-28T14:59:46,671,153,1,824,171,153,18,0.507556,53,0.507556,0.032122,Post-Industrial Estates / Deprived Working Com...


## 25c.6 Copy corrected pre-Adam review assets

In [7]:
# Save main caveat/confidence files into the v3 pre-Adam output folder.
save(high_conf, "pre_adam_high_confidence_top100_v3.csv")
save(medium_conf, "pre_adam_medium_confidence_top100_v3.csv")
save(serious_cav, "pre_adam_serious_caveat_top100_v3.csv")

# Party transition files, if available.
if party_all is not None:
    save(party_all, "pre_adam_party_transition_diagnostics_all_v3.csv")
if party_summary is not None:
    save(party_summary, "pre_adam_party_transition_summary_by_council_v3.csv")
if process_labels is not None:
    save(process_labels, "pre_adam_party_process_label_notes_v3.csv")

# Yorkshire files, if available.
if yorks_wards is not None:
    save(yorks_wards, "pre_adam_yorkshire_sdp_case_study_wards_v3.csv")
if yorks_tribe is not None:
    save(yorks_tribe, "pre_adam_yorkshire_sdp_performance_by_tribe_v3.csv")
if yorks_summary is not None:
    save(yorks_summary, "pre_adam_yorkshire_case_study_summary_v3.csv")
if middleton is not None:
    save(middleton, "pre_adam_middleton_park_case_study_profile_v3.csv")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_high_confidence_top100_v3.csv (671, 74)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_medium_confidence_top100_v3.csv (153, 74)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_serious_caveat_top100_v3.csv (1, 74)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_party_transition_diagnostics_all_v3.csv (825, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_party_transition_summary_by_council_v3.csv (35, 8)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_party_process_label_notes_v3.csv (3, 4)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_yorkshire_sdp_case_study_wards_v3

## 25c.7 Generate corrected revision notes

In [8]:
notes = f"""# Pre-Adam Report Revision Notes v3

## Current status

The North West structural model should now be presented as a breakthrough and organisational build model, not a council-control model and not a final target-seat list.

## Caveat treatment

The caveat language has been revised into confidence bands:

- High confidence: {len(high_conf)} rows.
- Medium confidence: {len(medium_conf)} rows.
- Serious caveat / manual review: {len(serious_cav)} rows.

This prevents technical mapping issues being presented as strategic warnings.

## SDP validation

The corrected SDP validation source is: `{sdp_path}`.

Current SDP validation rows: {len(sdp)}.
Matched to model: {int(sdp['matched_to_model_v2'].sum()) if 'matched_to_model_v2' in sdp.columns else 'unknown'}.
Unmatched rows: {len(unmatched)}.
Maximum observed SDP vote share: {num(sdp['sdp_vote_share_effective']).max():.1%}.

2026 data remains missing/provisional and should be added through a separate provisional file with mapping-confidence fields.

## Yorkshire case study

The Yorkshire case study remains the strongest empirical validation layer. It confirms that actual high SDP performance is concentrated in Post-Industrial / working-community wards, especially where paired with Settled Working Families.

## Party process labels

Use these process labels:

1. Conservative Legacy / Right-Adjacent Transition Terrain.
2. Labour Stronghold Breakthrough Terrain.
3. Reform / Independent Disruption Terrain.

The first is a diagnostic label, not proof of current Conservative-held opportunity.

## Report framing

The revised report should use three distinctions:

1. Breakthrough geography.
2. Build geography.
3. Control geography — not evidenced by Model v1 and requiring separate seat simulation.

The report should explicitly state that V1 does not forecast council control, vote share or seat totals.
"""
notes_path = OUTPUT_DIR / "pre_adam_report_revision_notes_v3.md"
notes_path.write_text(notes, encoding="utf-8")
print(notes_path)
print(notes)

c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_report_revision_notes_v3.md
# Pre-Adam Report Revision Notes v3

## Current status

The North West structural model should now be presented as a breakthrough and organisational build model, not a council-control model and not a final target-seat list.

## Caveat treatment

The caveat language has been revised into confidence bands:

- High confidence: 671 rows.
- Medium confidence: 153 rows.
- Serious caveat / manual review: 1 rows.

This prevents technical mapping issues being presented as strategic warnings.

## SDP validation

The corrected SDP validation source is: `c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v2\sdp_campaign_wards_profile_v2.csv`.

Current SDP validation rows: 171.
Matched to model: 153.
Unmatched rows: 18.
Maximum observed SDP vote share: 50.8%.

2026 data remains missing/provisional and should be added through a separate provisional f

## 25c.8 Manifest

In [9]:
manifest_rows = []
for p in sorted(OUTPUT_DIR.glob("pre_adam_*_v3.*")):
    manifest_rows.append({
        "file": p.name,
        "path": str(p),
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "note": "Generated by 25c with corrected SDP v2 source preference."
    })
manifest = pd.DataFrame(manifest_rows)
save(manifest, "pre_adam_revision_manifest_v3.csv")
manifest

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v3\pre_adam_revision_manifest_v3.csv (18, 4)


,file,path,created_at,note
0,pre_adam_headline_metrics_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
1,pre_adam_high_confidence_top100_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
2,pre_adam_medium_confidence_top100_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
3,pre_adam_middleton_park_case_study_profile_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
4,pre_adam_party_process_label_notes_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
5,pre_adam_party_transition_diagnostics_all_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
6,pre_adam_party_transition_summary_by_council_v...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
7,pre_adam_report_revision_notes_v3.md,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
8,pre_adam_sdp_candidate_count_check_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
9,pre_adam_sdp_highest_vote_share_cases_v3.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T14:59:46,Generated by 25c with corrected SDP v2 source ...
